# OMOP Observation Table

Transforms FHIR Observation resources (non-lab/vitals) into OMOP CDM `observation` table.

## Mapping: FHIR Observation → OMOP Observation

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| observation_id | Observation.id | Hash to integer |
| person_id | Observation.subject | Reference to person |
| observation_concept_id | Observation.code | Map to OMOP concept |
| observation_date | Observation.effectiveDateTime | Extract date |
| observation_datetime | Observation.effectiveDateTime | Full timestamp |
| observation_type_concept_id | - | 32817 (EHR) |
| value_as_number | Observation.valueQuantity.value | Numeric value |
| value_as_string | Observation.valueString | Text value |
| value_as_concept_id | Observation.valueCodeableConcept | Coded value |
| unit_concept_id | Observation.valueQuantity.unit | Unit concept |
| observation_source_value | Observation.code.coding[0].code | Original code |

## Categories Mapped to Observation

| FHIR Category | Description |
|---------------|-------------|
| social-history | Social determinants, smoking, etc. |
| survey | Questionnaire responses |
| exam | Physical exam findings |
| activity | Activity/exercise data |

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema();

## Create Observation Streaming Table

In [ ]:
DECLARE OR REPLACE VARIABLE create_observation_stmt STRING;

SET VARIABLE create_observation_stmt = "
CREATE OR REFRESH STREAMING TABLE observation (
  -- Primary key
  observation_id BIGINT NOT NULL COMMENT 'Unique observation identifier'
  
  -- Person reference
  ,person_id BIGINT NOT NULL COMMENT 'Reference to person table'
  
  -- Observation coding
  ,observation_concept_id INT NOT NULL COMMENT 'OMOP standard concept for observation'
  
  -- Dates
  ,observation_date DATE NOT NULL COMMENT 'Observation date'
  ,observation_datetime TIMESTAMP COMMENT 'Observation datetime'
  
  -- Type
  ,observation_type_concept_id INT NOT NULL DEFAULT 32817 COMMENT 'Type: 32817=EHR'
  
  -- Values
  ,value_as_number DECIMAL(18,6) COMMENT 'Numeric result value'
  ,value_as_string STRING COMMENT 'Text result value'
  ,value_as_concept_id INT DEFAULT 0 COMMENT 'Coded result value'
  
  -- Qualifier
  ,qualifier_concept_id INT DEFAULT 0 COMMENT 'Qualifier concept'
  ,qualifier_source_value STRING COMMENT 'Original qualifier value'
  
  -- Units
  ,unit_concept_id INT DEFAULT 0 COMMENT 'Unit concept'
  ,unit_source_value STRING COMMENT 'Original unit value'
  
  -- References
  ,provider_id BIGINT COMMENT 'Reference to provider table'
  ,visit_occurrence_id BIGINT COMMENT 'Reference to visit_occurrence table'
  ,visit_detail_id BIGINT COMMENT 'Reference to visit_detail table'
  
  -- Source values
  ,observation_source_value STRING COMMENT 'Original observation code'
  ,observation_source_concept_id INT DEFAULT 0 COMMENT 'Source vocabulary concept'
  
  -- Event reference
  ,observation_event_id BIGINT COMMENT 'Reference to event that caused observation'
  ,obs_event_field_concept_id INT DEFAULT 0 COMMENT 'Field concept for event reference'
  
  -- Additional info
  ,observation_code_system STRING COMMENT 'Source code system'
  ,observation_display STRING COMMENT 'Display text for observation'
  ,category STRING COMMENT 'FHIR category'
  
  -- Lineage
  ,fhir_observation_uuid STRING COMMENT 'Original FHIR Observation UUID'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Observation table - General observations from FHIR Observation resources'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer observation_id
  ABS(HASH(COALESCE(id::STRING, observation_uuid))) AS observation_id
  
  -- Person reference
  ,ABS(HASH(
    COALESCE(
      REGEXP_EXTRACT(subject:reference::STRING, 'Patient/(.+)', 1),
      subject:reference::STRING
    )
  )) AS person_id
  
  -- Observation concept - placeholder using hash
  ,COALESCE(
    ABS(HASH(code:coding[0]:code::STRING)) % 2000000000,
    0
  ) AS observation_concept_id
  
  -- Dates
  ,COALESCE(
    CAST(TRY_CAST(effectiveDateTime::STRING AS TIMESTAMP) AS DATE),
    CAST(TRY_CAST(effectivePeriod:start::STRING AS TIMESTAMP) AS DATE),
    CAST(TRY_CAST(issued::STRING AS TIMESTAMP) AS DATE),
    CURRENT_DATE()
  ) AS observation_date
  ,COALESCE(
    TRY_CAST(effectiveDateTime::STRING AS TIMESTAMP),
    TRY_CAST(effectivePeriod:start::STRING AS TIMESTAMP),
    TRY_CAST(issued::STRING AS TIMESTAMP)
  ) AS observation_datetime
  
  -- Type
  ,32817 AS observation_type_concept_id
  
  -- Values
  ,TRY_CAST(valueQuantity:value::STRING AS DECIMAL(18,6)) AS value_as_number
  ,COALESCE(
    valueString::STRING,
    valueCodeableConcept:text::STRING
  ) AS value_as_string
  ,CASE 
    WHEN valueCodeableConcept IS NOT NULL THEN
      ABS(HASH(valueCodeableConcept:coding[0]:code::STRING)) % 2000000000
    ELSE 0
  END AS value_as_concept_id
  
  -- Qualifier
  ,0 AS qualifier_concept_id
  ,NULL AS qualifier_source_value
  
  -- Units
  ,0 AS unit_concept_id
  ,COALESCE(valueQuantity:unit::STRING, valueQuantity:code::STRING) AS unit_source_value
  
  -- References
  ,CASE 
    WHEN performer[0]:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(performer[0]:reference::STRING, 'Practitioner/(.+)', 1)))
    ELSE NULL
  END AS provider_id
  ,CASE 
    WHEN encounter:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(encounter:reference::STRING, 'Encounter/(.+)', 1)))
    ELSE NULL
  END AS visit_occurrence_id
  ,NULL AS visit_detail_id
  
  -- Source values
  ,code:coding[0]:code::STRING AS observation_source_value
  ,0 AS observation_source_concept_id
  
  -- Event reference
  ,NULL AS observation_event_id
  ,0 AS obs_event_field_concept_id
  
  -- Additional info
  ,code:coding[0]:system::STRING AS observation_code_system
  ,COALESCE(code:coding[0]:display::STRING, code:text::STRING) AS observation_display
  ,category[0]:coding[0]:code::STRING AS category
  
  -- Lineage
  ,observation_uuid AS fhir_observation_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".observation)
WHERE status::STRING IN ('final', 'amended', 'corrected', 'preliminary')
  AND (
    category[0]:coding[0]:code::STRING NOT IN ('laboratory', 'vital-signs')
    OR category[0]:coding[0]:code IS NULL
  )
  AND code:coding[0]:system::STRING NOT LIKE '%loinc%'
";

SELECT create_observation_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_observation_stmt;

In [ ]:
-- Verify observation table
SELECT 
  observation_id,
  person_id,
  observation_concept_id,
  observation_date,
  value_as_number,
  value_as_string,
  observation_source_value,
  observation_display,
  category
FROM observation
LIMIT 10;

In [ ]:
-- Observations by category
SELECT 
  category,
  COUNT(*) AS count
FROM observation
GROUP BY category
ORDER BY count DESC;

In [ ]:
-- Top observations
SELECT 
  observation_source_value,
  observation_display,
  COUNT(*) AS occurrences
FROM observation
GROUP BY observation_source_value, observation_display
ORDER BY occurrences DESC
LIMIT 20;